# Import

In [1]:
!pip -q install transformers accelerate datasets sentencepiece

# Model

In [2]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
# 더 가볍게 시작하려면:
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

print("Loaded:", MODEL_NAME)
print("Device:", model.device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-3B-Instruct
Device: cuda:0


# Util Func

In [3]:
import re

def normalize_text(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def clean_answer(text: str) -> str:
    text = text.strip()
    # 첫 줄만 사용
    text = text.split("\n")[0].strip()
    # 코드블록 같은 이상 출력 잘라내기
    text = text.split("```")[0].strip()
    return text

def generate_text(prompt: str, max_new_tokens: int = 128):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # temperature/top_p/top_k 안 씀
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded

## QA Func

### With Context

In [4]:
def ask_qa_with_context(context: str, question: str, max_new_tokens: int = 48):
    prompt = f"""다음 문서를 읽고 질문에 답하세요.

규칙:
1. 답만 한 문장으로 짧게 출력
2. 설명하지 말 것
3. 코드블록 출력 금지

[문서]
{context}

[질문]
{question}

[답변]
"""
    text = generate_text(prompt, max_new_tokens=max_new_tokens)
    if "[답변]" in text:
        text = text.split("[답변]")[-1].strip()
    return clean_answer(text)



### Without Context

In [5]:
def ask_qa_without_context(question: str, max_new_tokens: int = 48):
    prompt = f"""다음 질문에 답하세요.

규칙:
1. 답만 한 문장으로 짧게 출력
2. 설명하지 말 것
3. 코드블록 출력 금지

[질문]
{question}

[답변]
"""
    text = generate_text(prompt, max_new_tokens=max_new_tokens)
    if "[답변]" in text:
        text = text.split("[답변]")[-1].strip()
    return clean_answer(text)

## Predict Search Func

In [6]:
def predict_need_search(question: str, context: str = "", max_new_tokens: int = 4):
    prompt = f"""당신의 임무는 외부 검색 필요 여부만 판단하는 것입니다.

규칙:
1. 최신 정보, 현재 시점 정보, 자주 바뀌는 사실이면 SEARCH
2. 일반 상식, 오래 변하지 않는 사실이면 NO_SEARCH
3. 반드시 SEARCH 또는 NO_SEARCH 중 하나만 출력
4. 설명 금지

[질문]
{question}

[문맥]
{context if context else "없음"}

[판단]
"""
    text = generate_text(prompt, max_new_tokens=max_new_tokens)

    if "[판단]" in text:
        result = text.split("[판단]")[-1].strip()
    else:
        result = text.strip()

    result = result.split()[0].upper()

    if result == "SEARCH":
        return "SEARCH"
    elif result == "NO_SEARCH":
        return "NO_SEARCH"
    return result

# DataSet

In [7]:
from datasets import load_dataset

dataset = load_dataset("KorQuAD/squad_kor_v1", split="validation[:300]")

print(type(dataset))
print(dataset)

README.md: 0.00B [00:00, ?B/s]

squad_kor_v1/train-00000-of-00001.parque(…):   0%|          | 0.00/11.6M [00:00<?, ?B/s]

squad_kor_v1/validation-00000-of-00001.p(…):   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60407 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5774 [00:00<?, ? examples/s]

<class 'datasets.arrow_dataset.Dataset'>
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 300
})


In [8]:
print("=== Dataset Summary ===")
print(dataset)

print("\n=== Number of Rows ===")
print(len(dataset))

print("\n=== Column Names ===")
print(dataset.column_names)

print("\n=== First Sample ===")
print(dataset[0])

=== Dataset Summary ===
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 300
})

=== Number of Rows ===
300

=== Column Names ===
['id', 'title', 'context', 'question', 'answers']

=== First Sample ===
{'id': '6548850-0-0', 'title': '임종석', 'context': '1989년 2월 15일 여의도 농민 폭력 시위를 주도한 혐의(폭력행위등처벌에관한법률위반)으로 지명수배되었다. 1989년 3월 12일 서울지방검찰청 공안부는 임종석의 사전구속영장을 발부받았다. 같은 해 6월 30일 평양축전에 임수경을 대표로 파견하여 국가보안법위반 혐의가 추가되었다. 경찰은 12월 18일~20일 사이 서울 경희대학교에서 임종석이 성명 발표를 추진하고 있다는 첩보를 입수했고, 12월 18일 오전 7시 40분 경 가스총과 전자봉으로 무장한 특공조 및 대공과 직원 12명 등 22명의 사복 경찰을 승용차 8대에 나누어 경희대학교에 투입했다. 1989년 12월 18일 오전 8시 15분 경 서울청량리경찰서는 호위 학생 5명과 함께 경희대학교 학생회관 건물 계단을 내려오는 임종석을 발견, 검거해 구속을 집행했다. 임종석은 청량리경찰서에서 약 1시간 동안 조사를 받은 뒤 오전 9시 50분 경 서울 장안동의 서울지방경찰청 공안분실로 인계되었다.', 'question': '임종석이 여의도 농민 폭력 시위를 주도한 혐의로 지명수배 된 날은?', 'answers': {'text': ['1989년 2월 15일'], 'answer_start': [0]}}


## rough match Func

In [9]:
def rough_match(pred: str, gold: str) -> bool:
    pred_n = normalize_text(pred)
    gold_n = normalize_text(gold)
    return (gold_n in pred_n) or (pred_n in gold_n)

## PipeLine

In [10]:
def qa_with_search_pipeline(question, context):
    decision = predict_need_search(question, context)

    if decision == "SEARCH":
        pred = ask_qa_with_context(context, question)
    else:
        pred = ask_qa_without_context(question)

    return decision, pred

### PipeLine Test

In [11]:
pipeline_correct = 0
pipeline_results = []

for ex in dataset:
    question = ex["question"]
    context = ex["context"]
    gold_answers = ex["answers"]["text"]
    gold = gold_answers[0] if len(gold_answers) > 0 else ""

    decision, pred = qa_with_search_pipeline(question, context)

    ok = rough_match(pred, gold)
    pipeline_correct += int(ok)

    pipeline_results.append({
        "question": question,
        "decision": decision,
        "gold": gold,
        "pred": pred,
        "correct": ok
    })

pipeline_acc = pipeline_correct / len(dataset)
print(f"Pipeline accuracy: {pipeline_correct}/{len(dataset)} = {pipeline_acc:.3f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Pipeline accuracy: 250/300 = 0.833


### PipeLine Errors

In [12]:
pipeline_errors = [r for r in pipeline_results if not r["correct"]]
print(f"Pipeline wrong cases: {len(pipeline_errors)}")

for i, case in enumerate(pipeline_errors[:10]):
    print(f"\n[Pipeline Error {i+1}]")
    print("Q:", case["question"])
    print("Decision:", case["decision"])
    print("Gold:", case["gold"])
    print("Pred:", case["pred"])

Pipeline wrong cases: 50

[Pipeline Error 1]
Q: 임종석을 검거한 장소는 경희대 내 어디인가?
Decision: SEARCH
Gold: 학생회관 건물 계단
Pred: 학생회관 계단

[Pipeline Error 2]
Q: 미국 군대 내 두번째로 높은 직위는 무엇인가?
Decision: SEARCH
Gold: 미국 육군 부참모 총장
Pred: 美军参谋长联席会议主席

[Pipeline Error 3]
Q: 미국 군대에서 두번째로 높은 직위는?
Decision: SEARCH
Gold: 미국 육군 부참모 총장
Pred: 장교 대장

[Pipeline Error 4]
Q: 알렉산더 헤이그가 미국 육군사관학교로 임명받은 해는 언제인가?
Decision: SEARCH
Gold: 1944년
Pred: 1947년

[Pipeline Error 5]
Q: 헤이그가 공부한 대학교는?
Decision: NO_SEARCH
Gold: 노터데임 대학교
Pred: 미국 스탠퍼드 대

[Pipeline Error 6]
Q: 알렉산더 헤이그가 나온 대학교는?
Decision: NO_SEARCH
Gold: 노터데임 대학교
Pred: 约翰斯·霍普金斯大学

[Pipeline Error 7]
Q: 육군사관학교에서 졸업한 헤이그가 제일 처음 소위로 발령받은 부대는 무엇이었나?
Decision: SEARCH
Gold: 정통 제병 연합부대
Pred: 캔자스 주 포트라일리

[Pipeline Error 8]
Q: 제럴드 포드 대통령 시기 헤이그가 최고사령부의 최고 사령관으로 임명된 곳은 어디인가?
Decision: SEARCH
Gold: 미국 유럽 연합군
Pred: 나토에서

[Pipeline Error 9]
Q: 퇴역 후 헤이그는 어느 회사의 대표가 되었나?
Decision: SEARCH
Gold: 미국 기술 주식 회사
Pred: AMC 엔터테인먼트

[Pipeline Error 10]
Q: 노아의 방주에 대해 기록하고있는 복음서는 무엇인가?
Decision: NO_S

In [13]:
under_search = []
qa_failure = []
over_search = []

for r in pipeline_results:
    if not r["correct"]:
        if r["decision"] == "NO_SEARCH":
            under_search.append(r)
        elif r["decision"] == "SEARCH":
            qa_failure.append(r)

print("UNDER_SEARCH:", len(under_search))
print("QA_FAILURE:", len(qa_failure))

UNDER_SEARCH: 15
QA_FAILURE: 35


## 오류 라벨링

In [15]:
from google.colab import drive
import pandas as pd
import os

# 1. 드라이브 연결
drive.mount('/content/drive')

# 2. pipeline 오답 추출
pipeline_errors = []
for idx, r in enumerate(pipeline_results):
    if not r["correct"]:
        pipeline_errors.append({
            "dataset_idx": int(idx),
            **r
        })

print("Total pipeline errors:", len(pipeline_errors))

# 3. 분석할 개수 설정
N = min(20, len(pipeline_errors))
analysis_samples = pipeline_errors[:N]

# 4. 최종 라벨링용 테이블 생성 (context 전체 포함)
analysis_table = pd.DataFrame([
    {
        "row_id": i,
        "dataset_idx": int(case["dataset_idx"]),
        "question": case["question"],
        "decision": case["decision"],
        "gold": case["gold"],
        "pred": case["pred"],
        "all_gold_answers": " | ".join(dataset[int(case["dataset_idx"])]["answers"]["text"]),
        "context_full": dataset[int(case["dataset_idx"])]["context"],   # ✅ 전체 context
        "retrieval_quality": "",   # YES / PARTIAL / NO
        "utilization_type": "",    # IGNORED / PARTIAL_USE / CONTRADICTED / HALLUCINATED / OTHER
        "notes": ""
    }
    for i, case in enumerate(analysis_samples)
])

# 5. 저장 경로
local_path = "./repro_failure_analysis_full_context.csv"
drive_path = "/content/drive/MyDrive/retrieval-utilization/repro/repro_02_failure_analysis_full_context.csv"

# 6. 저장
analysis_table.to_csv(local_path, index=False, encoding="utf-8-sig")
analysis_table.to_csv(drive_path, index=False, encoding="utf-8-sig")

# 7. 확인
print("\n✅ 저장 완료")
print("현재 폴더:", os.getcwd())
print("로컬 저장:", os.path.abspath(local_path))
print("드라이브 저장:", drive_path)
print("\n현재 폴더 파일 목록:")
print(os.listdir("."))

# 8. 미리보기
display(analysis_table.head(3))

Mounted at /content/drive
Total pipeline errors: 50

✅ 저장 완료
현재 폴더: /content
로컬 저장: /content/repro_failure_analysis_full_context.csv
드라이브 저장: /content/drive/MyDrive/retrieval-utilization/repro/repro_02_failure_analysis_full_context.csv

현재 폴더 파일 목록:
['.config', 'repro_failure_analysis_full_context.csv', 'drive', 'sample_data']


,row_id,dataset_idx,question,decision,gold,pred,all_gold_answers,context_full,retrieval_quality,utilization_type,notes
0,0,3,임종석을 검거한 장소는 경희대 내 어디인가?,SEARCH,학생회관 건물 계단,학생회관 계단,학생회관 건물 계단,1989년 2월 15일 여의도 농민 폭력 시위를 주도한 혐의(폭력행위등처벌에관한법률...,,,
1,1,11,미국 군대 내 두번째로 높은 직위는 무엇인가?,SEARCH,미국 육군 부참모 총장,美军参谋长联席会议主席,미국 육군 부참모 총장,"알렉산더 메이그스 헤이그 2세(영어: Alexander Meigs Haig, Jr....",,,
2,2,15,미국 군대에서 두번째로 높은 직위는?,SEARCH,미국 육군 부참모 총장,장교 대장,미국 육군 부참모 총장,"알렉산더 메이그스 헤이그 2세(영어: Alexander Meigs Haig, Jr....",,,


### 라밸링 기준
#### retrieval_quality
	•	YES: context 안에 정답 단서가 충분히 있음
	•	PARTIAL: 단서는 있는데 불완전하거나 간접적
	•	NO: context 안에 정답 단서가 거의 없음

#### utilization_type
	•	IGNORED: 답이 있었는데 모델이 안 씀
	•	PARTIAL_USE: 일부만 참고해서 틀림
	•	CONTRADICTED: 문서와 반대로 답함 / 다른 entity 선택
	•	HALLUCINATED: 문서와 무관한 답 생성
	•	OTHER: 애매한 경우

In [16]:
analysis_table.loc[0, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "almost correct but shortened exact phrase: '학생회관 계단' vs '학생회관 건물 계단'"
]
analysis_table.loc[1, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context explicitly states '미국 육군 부참모 총장'; model outputs another title in Chinese"
]
analysis_table.loc[2, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "correct position is explicit, but model outputs generic military rank"
]
analysis_table.loc[3, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context contains both 1944 appointment and 1947 graduation; model chose wrong year"
]
analysis_table.loc[4, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "IGNORED", "first sentence explicitly contains '노터데임 대학교'; model ignored evidence after NO_SEARCH"
]
analysis_table.loc[5, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "IGNORED", "same pattern as row 4"
]
analysis_table.loc[6, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "question asks for unit, but model answers location"
]
analysis_table.loc[7, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "model partially uses context but returns location/institution-like phrase instead of exact target"
]
analysis_table.loc[8, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context explicitly states '미국 기술 주식 회사'; model outputs unrelated company"
]
analysis_table.loc[9, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "IGNORED", "context explicitly mentions 창세기; model hallucinates gospel names"
]
analysis_table.loc[10, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context gives 1 cubit as 45~46cm, but model uses 300 cubits ≈ 135m"
]
analysis_table.loc[11, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "close orthographic variant of correct material, not exact answer"
]
analysis_table.loc[12, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "meaning is close but answer is longer and less exact than gold"
]
analysis_table.loc[13, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "correct group is present but model generates verbose answer with noise"
]
analysis_table.loc[14, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "IGNORED", "context explicitly states '역사책'; model outputs meaningless '네'"
]
analysis_table.loc[15, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "semantically close but longer descriptive answer than gold"
]
analysis_table.loc[16, ["retrieval_quality", "utilization_type", "notes"]] = [
    "PARTIAL", "HALLUCINATED", "in the shown context snippet BTV evidence is not visible; model hallucinates CCTV"
]
analysis_table.loc[17, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context implies only a few thousand years, but model outputs much larger timescale"
]
analysis_table.loc[18, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "answer is semantically close but less exact than gold"
]
analysis_table.loc[19, ["retrieval_quality", "utilization_type", "notes"]] = [
    "PARTIAL", "CONTRADICTED", "visible context snippet does not show 375 days; model likely latched onto another number in passage"
]

analysis_table.head(20)

,row_id,dataset_idx,question,decision,gold,pred,all_gold_answers,context_full,retrieval_quality,utilization_type,notes
0,0,3,임종석을 검거한 장소는 경희대 내 어디인가?,SEARCH,학생회관 건물 계단,학생회관 계단,학생회관 건물 계단,1989년 2월 15일 여의도 농민 폭력 시위를 주도한 혐의(폭력행위등처벌에관한법률...,YES,PARTIAL_USE,almost correct but shortened exact phrase: '학생...
1,1,11,미국 군대 내 두번째로 높은 직위는 무엇인가?,SEARCH,미국 육군 부참모 총장,美军参谋长联席会议主席,미국 육군 부참모 총장,"알렉산더 메이그스 헤이그 2세(영어: Alexander Meigs Haig, Jr....",YES,CONTRADICTED,context explicitly states '미국 육군 부참모 총장'; mode...
2,2,15,미국 군대에서 두번째로 높은 직위는?,SEARCH,미국 육군 부참모 총장,장교 대장,미국 육군 부참모 총장,"알렉산더 메이그스 헤이그 2세(영어: Alexander Meigs Haig, Jr....",YES,CONTRADICTED,"correct position is explicit, but model output..."
3,3,20,알렉산더 헤이그가 미국 육군사관학교로 임명받은 해는 언제인가?,SEARCH,1944년,1947년,1944년,노터데임 대학교에서 2년간 합리적으로 심각한 공부를 한 후 헤이그는 1944년 미국...,YES,CONTRADICTED,context contains both 1944 appointment and 194...
4,4,22,헤이그가 공부한 대학교는?,NO_SEARCH,노터데임 대학교,미국 스탠퍼드 대,노터데임 대학교,노터데임 대학교에서 2년간 합리적으로 심각한 공부를 한 후 헤이그는 1944년 미국...,YES,IGNORED,first sentence explicitly contains '노터데임 대학교';...
5,5,24,알렉산더 헤이그가 나온 대학교는?,NO_SEARCH,노터데임 대학교,约翰斯·霍普金斯大学,노터데임 대학교,노터데임 대학교에서 2년간 합리적으로 심각한 공부를 한 후 헤이그는 1944년 미국...,YES,IGNORED,same pattern as row 4
6,6,25,육군사관학교에서 졸업한 헤이그가 제일 처음 소위로 발령받은 부대는 무엇이었나?,SEARCH,정통 제병 연합부대,캔자스 주 포트라일리,정통 제병 연합부대,노터데임 대학교에서 2년간 합리적으로 심각한 공부를 한 후 헤이그는 1944년 미국...,YES,CONTRADICTED,"question asks for unit, but model answers loca..."
7,7,31,제럴드 포드 대통령 시기 헤이그가 최고사령부의 최고 사령관으로 임명된 곳은 어디인가?,SEARCH,미국 유럽 연합군,나토에서,미국 유럽 연합군,헤이그는 닉슨 대통령이 그를 사성 장군과 육군 부참모로 진급시킬 때 집중 광선과 논...,YES,PARTIAL_USE,model partially uses context but returns locat...
8,8,33,퇴역 후 헤이그는 어느 회사의 대표가 되었나?,SEARCH,미국 기술 주식 회사,AMC 엔터테인먼트,미국 기술 주식 회사,헤이그는 닉슨 대통령이 그를 사성 장군과 육군 부참모로 진급시킬 때 집중 광선과 논...,YES,CONTRADICTED,context explicitly states '미국 기술 주식 회사'; model...
9,9,41,노아의 방주에 대해 기록하고있는 복음서는 무엇인가?,NO_SEARCH,창세기,"마태복음, 누리복음, 마르키움복음",창세기,"노아는 하나님의 명령에 따라 배를 만들고 가족과 정결한 짐승 암수 일곱 마리씩, 부...",YES,IGNORED,context explicitly mentions 창세기; model halluci...


In [17]:
print("=== Retrieval Quality Counts ===")
print(analysis_table["retrieval_quality"].value_counts())

print("\n=== Utilization Type Counts ===")
print(analysis_table["utilization_type"].value_counts())

=== Retrieval Quality Counts ===
retrieval_quality
YES        18
PARTIAL     2
Name: count, dtype: int64

=== Utilization Type Counts ===
utilization_type
CONTRADICTED    8
PARTIAL_USE     7
IGNORED         4
HALLUCINATED    1
Name: count, dtype: int64
